In [1]:
# Relative paths - portable across machines after cloning the repository
from pathlib import Path

DATA_RAW       = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")


---

## Airport Reviews — Cleaning

*Source notebook: `notebook_airport_reviews.ipynb`*


In [3]:
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pandas as pd
import pycountry

# ==============================================================================
# BƯỚC 0: TẢI DỮ LIỆU & CHUẨN BỊ THƯ VIỆN NLP
# ==============================================================================
# Tải bộ từ điển cảm xúc của NLTK (chỉ cần chạy 1 lần trên máy)
nltk.download("vader_lexicon", quiet=True)
sia = SentimentIntensityAnalyzer()

# Đọc file dữ liệu sân bay gốc
df = pd.read_csv(DATA_RAW / "airport_reviews.csv")
print(f"Kích thước file gốc: {df.shape}")

# 1. Tập Stopwords tiếng Anh nền tảng
base_stopwords = {
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "your",
    "yours",
    "he",
    "him",
    "his",
    "she",
    "her",
    "it",
    "its",
    "they",
    "them",
    "their",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "can",
    "will",
    "just",
    "don",
    "should",
    "now",
}

# 2. Từ điển Stopwords ĐẶC THÙ CHO SÂN BAY & từ trung tính gây nhiễu Word Cloud
airport_noise_words = {
    "airport",
    "airports",
    "terminal",
    "terminals",
    "gate",
    "gates",
    "flight",
    "flights",
    "fly",
    "flying",
    "plane",
    "planes",
    "area",
    "areas",
    "place",
    "places",
    "time",
    "times",
    "hour",
    "hours",
    "minute",
    "minutes",
    "passenger",
    "passengers",
    "people",
    "traveller",
    "travellers",
    "would",
    "could",
    "get",
    "got",
    "getting",
    "one",
    "two",
    "three",
    "also",
    "even",
    "really",
    "first",
    "second",
    "day",
    "days",
    "bit",
    "much",
    "give",
    "gave",
    "see",
    "saw",
    "seen",
    "us",
    "take",
    "took",
    "taking",
    "around",
    "since",
    "another",
    "someone",
    "anyone",
    "something",
    "anything",
    "nothing",
    "everything",
    "say",
    "said",
    "told",
    "ask",
    "asked",
    "went",
    "go",
    "going",
    "come",
    "came",
    "back",
    "next",
    "last",
    "many",
    "lot",
    "lots",
    "little",
    "small",
    "big",
    "know",
    "think",
    "thought",
    "want",
    "wanted",
    "need",
    "needed",
    "use",
    "used",
    "using",
    "try",
    "tried",
    "still",
    "like",
    "quite",
    "well",
    "way",
    "make",
    "made",
    "good",
    "bad",  # Bỏ good/bad chung chung để làm nổi bật từ cụ thể: rude, dirty, efficient, crowded, fast
}
stop_words = base_stopwords.union(airport_noise_words)

# 3. Chuẩn bị tập từ điển quốc gia chuẩn ISO
valid_countries = {c.name.lower() for c in pycountry.countries}
for c in pycountry.countries:
    if hasattr(c, "official_name"):
        valid_countries.add(c.official_name.lower())
    if hasattr(c, "common_name"):
        valid_countries.add(c.common_name.lower())
custom_countries = {
    "uk",
    "usa",
    "united states",
    "united kingdom",
    "uae",
    "russia",
    "south korea",
    "north korea",
    "vietnam",
    "laos",
    "taiwan",
    "hong kong",
    "macau",
}
valid_countries = valid_countries.union(custom_countries)

Kích thước file gốc: (49505, 20)


In [4]:
# ==============================================================================
# BƯỚC 1: XÓA CỘT THỪA & CHUYỂN ĐỔI DATA TYPE
# ==============================================================================
print("1. Đang dọn dẹp các cột thừa và định dạng kiểu dữ liệu...")
cols_to_drop = ["customer_name", "updated_at", "date_visit"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Chuyển kiểu ngày tháng
df["date_submitted"] = pd.to_datetime(df["date_submitted"], errors="coerce")

# Chuẩn hóa 8 cột điểm số dịch vụ sân bay về kiểu số Numeric
rating_cols = [
    "queuing_times",
    "terminal_cleanliness",
    "terminal_seating",
    "terminal_signs",
    "food_beverages",
    "airport_shopping",
    "airport_staff",
    "wifi_connectivity",
    "value_for_money",
    "recommended",
]
for col in rating_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")



1. Đang dọn dẹp các cột thừa và định dạng kiểu dữ liệu...


In [5]:


# ==============================================================================
# BƯỚC 2: TẠO CỘT PHÂN CHIA GIAI ĐOẠN DỮ LIỆU (DATA ERA)
# ==============================================================================
df["data_era"] = df["date_submitted"].apply(
    lambda x: "Modern (2015-Present)"
    if pd.notnull(x) and x.year >= 2015
    else "Historical (Pre-2015)"
)




In [6]:
# ==============================================================================
# BƯỚC 3: CHUẨN HÓA QUỐC TỊCH (NATIONALITY) & TRẢI NGHIỆM (EXPERIENCE)
# ==============================================================================
def clean_nationality(val):
    if pd.isna(val):
        return "Unknown"
    clean_val = str(val).strip().lower()
    return str(val).strip().title() if clean_val in valid_countries else "Unknown"


print("2. Đang chuẩn hóa quốc tịch và trải nghiệm tại sân bay...")
if "nationality" in df.columns:
    df["nationality"] = df["nationality"].apply(clean_nationality)

# Chuẩn hóa khoảng trắng thừa trong cột experience_at_airport
if "experience_at_airport" in df.columns:
    df["experience_at_airport"] = (
        df["experience_at_airport"].astype(str).str.strip().replace("nan", "Unknown")
    )

2. Đang chuẩn hóa quốc tịch và trải nghiệm tại sân bay...


In [7]:
# ==============================================================================
# BƯỚC 4: PHÂN TÍCH CẢM XÚC TRÊN TEXT GỐC (VADER SENTIMENT)
# ==============================================================================
print("3. Đang chấm điểm cảm xúc (Sentiment Analysis) cho review sân bay...")


def get_sentiment(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Neutral", 0.0
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "Positive", score
    elif score <= -0.05:
        return "Negative", score
    else:
        return "Neutral", score


sentiment_res = df["review"].apply(get_sentiment)
df["sentiment_label"] = [res[0] for res in sentiment_res]
df["sentiment_score"] = [res[1] for res in sentiment_res]



3. Đang chấm điểm cảm xúc (Sentiment Analysis) cho review sân bay...


In [8]:
# ==============================================================================
# BƯỚC 5: TÁCH TỪ KHÓA SIÊU SẠCH (AIRPORT KEYWORDS) & XÓA TEXT GỐC
# ==============================================================================
print("4. Đang trích xuất từ khóa cho Word Cloud và tối ưu dung lượng...")


def extract_airport_keywords(text):
    if pd.isna(text):
        return ""
    words = re.findall(r"\b[a-z]{3,}\b", str(text).lower())
    keywords = [w for w in words if w not in stop_words]
    return " ".join(keywords)


df["clean_keywords"] = df["review"].apply(extract_airport_keywords)

# [QUAN TRỌNG NHẤT]: Xóa cột review gốc để giảm 80% dung lượng cho Power BI
df = df.drop(columns=["review"])



4. Đang trích xuất từ khóa cho Word Cloud và tối ưu dung lượng...


In [9]:
# ==============================================================================
# BƯỚC 6: XUẤT FILE HOÀN THIỆN CHO POWER BI
# ==============================================================================
output_filename = "airport_reviews_PBI_Master_Optimized.csv"
df.to_csv(output_filename, index=False)
print("-" * 60)
print(f"XỬ LÝ HOÀN TẤT! File sẵn sàng cho Power BI: '{output_filename}'")
print(f"Kích thước cuối cùng: {df.shape}")
print("-" * 60)

------------------------------------------------------------
XỬ LÝ HOÀN TẤT! File sẵn sàng cho Power BI: 'airport_reviews_PBI_Master_Optimized.csv'
Kích thước cuối cùng: (49505, 20)
------------------------------------------------------------


In [10]:
def final_quality_check(df, file_name):
    print(f"=== KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG: {file_name} ===")

    # 1. Kiểm tra xem còn cột nào bị NULL/NaN không
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if not null_cols.empty:
        print("⚠️ Cảnh báo: Vẫn còn cột chứa giá trị NULL:")
        for col, count in null_cols.items():
            print(f"   - {col}: trống {count} dòng ({count/len(df)*100:.1f}%)")
    else:
        print("✔️ Tuyệt vời! Không còn ô NULL nào trong dữ liệu.")

    # 2. Kiểm tra kiểu dữ liệu của date_submitted và recommended
    if "date_submitted" in df.columns:
        print(f"✔️ Kiểu dữ liệu ngày tháng: {df['date_submitted'].dtype}")
    if "recommended" in df.columns:
        print(
            f"✔️ Kiểu dữ liệu NPS (recommended): {df['recommended'].dtype} (Giá trị mẫu: {df['recommended'].unique()})"
        )
    print("-" * 50)


# Ví dụ cách dùng:
final_quality_check(df, "Airports Reviews")


=== KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG: Airports Reviews ===
⚠️ Cảnh báo: Vẫn còn cột chứa giá trị NULL:
   - type_of_traveller: trống 17271 dòng (34.9%)
   - queuing_times: trống 5218 dòng (10.5%)
   - terminal_cleanliness: trống 5306 dòng (10.7%)
   - terminal_seating: trống 21413 dòng (43.3%)
   - terminal_signs: trống 17919 dòng (36.2%)
   - food_beverages: trống 24202 dòng (48.9%)
   - airport_shopping: trống 14366 dòng (29.0%)
   - airport_staff: trống 20009 dòng (40.4%)
   - wifi_connectivity: trống 27729 dòng (56.0%)
✔️ Kiểu dữ liệu ngày tháng: datetime64[ns]
✔️ Kiểu dữ liệu NPS (recommended): int64 (Giá trị mẫu: [1 0])
--------------------------------------------------


In [11]:
# 1. Vá lỗi cột text (Biến NULL thành Unknown)
df["type_of_traveller"] = df["type_of_traveller"].fillna("Unknown")

# 2. Xóa các dòng rác (nếu có) bị trống toàn bộ 8 cột điểm số (Khách không chấm điểm nào)
# Bước này tùy chọn: Giúp file nhẹ hơn vì các dòng này không vẽ được biểu đồ Radar/Bar chart
rating_cols = [
    "queuing_times", "terminal_cleanliness", "terminal_seating", 
    "terminal_signs", "food_beverages", "airport_shopping", 
    "airport_staff", "wifi_connectivity"
]
# Giữ lại review nếu khách có chấm ít nhất 1 trong 8 tiêu chí trên
df = df.dropna(subset=rating_cols, how='all')

# 3. Chạy lại hàm kiểm tra
print("Đã vá lỗi xong! Tình trạng cột type_of_traveller hiện tại:")
print(f"Số ô NULL: {df['type_of_traveller'].isnull().sum()}")

# Xuất lại đè lên file cũ
df.to_csv("airport_reviews_PBI_Master_Optimized.csv", index=False)
print("Đã xuất lại file chuẩn 100% cho Power BI!")

Đã vá lỗi xong! Tình trạng cột type_of_traveller hiện tại:
Số ô NULL: 0
Đã xuất lại file chuẩn 100% cho Power BI!


---

## Lounge Reviews — Cleaning & Error Handling

*Source notebook: `notebook_lounge_review_errors.ipynb`*


In [13]:
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pandas as pd
import pycountry

# ==============================================================================
# BƯỚC 0: TẢI DỮ LIỆU & CHUẨN BỊ THƯ VIỆN NLP
# ==============================================================================
nltk.download("vader_lexicon", quiet=True)
sia = SentimentIntensityAnalyzer()

# Đọc file dữ liệu đánh giá phòng chờ
df = pd.read_csv(DATA_RAW / "lounge_reviews.csv")
print(f"Kích thước file gốc: {df.shape}")

# 1. Từ điển Stopwords tiếng Anh nền tảng
base_stopwords = {
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "your",
    "yours",
    "he",
    "him",
    "his",
    "she",
    "her",
    "it",
    "its",
    "they",
    "them",
    "their",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "can",
    "will",
    "just",
    "don",
    "should",
    "now",
}

# 2. Từ điển Stopwords ĐẶC THÙ CHO PHÒNG CHỜ (Lounge-specific Stopwords)
lounge_noise_words = {
    "lounge",
    "lounges",
    "room",
    "rooms",
    "area",
    "areas",
    "seat",
    "seats",
    "seated",
    "seating",
    "airport",
    "airports",
    "terminal",
    "terminals",
    "flight",
    "flights",
    "fly",
    "flying",
    "flew",
    "plane",
    "planes",
    "airline",
    "airlines",
    "time",
    "times",
    "hour",
    "hours",
    "minute",
    "minutes",
    "passenger",
    "passengers",
    "people",
    "traveller",
    "travellers",
    "would",
    "could",
    "get",
    "got",
    "getting",
    "one",
    "two",
    "three",
    "also",
    "even",
    "really",
    "first",
    "second",
    "day",
    "days",
    "bit",
    "much",
    "give",
    "gave",
    "see",
    "saw",
    "seen",
    "us",
    "take",
    "took",
    "taking",
    "around",
    "since",
    "another",
    "someone",
    "anyone",
    "something",
    "anything",
    "nothing",
    "everything",
    "say",
    "said",
    "told",
    "ask",
    "asked",
    "went",
    "go",
    "going",
    "come",
    "came",
    "back",
    "next",
    "last",
    "many",
    "lot",
    "lots",
    "little",
    "small",
    "big",
    "know",
    "think",
    "thought",
    "want",
    "wanted",
    "need",
    "needed",
    "use",
    "used",
    "using",
    "try",
    "tried",
    "still",
    "like",
    "quite",
    "well",
    "way",
    "make",
    "made",
    "good",
    "bad",  # Bỏ good/bad chung chung để lòi ra: crowded, delicious, champagne, dirty, quiet, relaxing
}
stop_words = base_stopwords.union(lounge_noise_words)

# 3. Chuẩn bị tập từ điển quốc gia chuẩn ISO
valid_countries = {c.name.lower() for c in pycountry.countries}
for c in pycountry.countries:
    if hasattr(c, "official_name"):
        valid_countries.add(c.official_name.lower())
    if hasattr(c, "common_name"):
        valid_countries.add(c.common_name.lower())
custom_countries = {
    "uk",
    "usa",
    "united states",
    "united kingdom",
    "uae",
    "russia",
    "south korea",
    "north korea",
    "vietnam",
    "laos",
    "taiwan",
    "hong kong",
    "macau",
}
valid_countries = valid_countries.union(custom_countries)



Kích thước file gốc: (5087, 21)


In [14]:
# ==============================================================================
# BƯỚC 1: XÓA CỘT THỪA & ĐỔI DATA TYPE
# ==============================================================================
print("1. Đang dọn dẹp cột thừa và chuẩn hóa kiểu dữ liệu...")
cols_to_drop = ["customer_name", "updated_at", "date_visit"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Chuẩn hóa ngày tháng
df["date_submitted"] = pd.to_datetime(df["date_submitted"], errors="coerce")

# Chuẩn hóa TOÀN BỘ 7 cột điểm số phòng chờ + recommended về kiểu Numeric
rating_cols = [
    "comfort",
    "cleanliness",
    "bar_and_beverages",
    "catering",
    "washrooms",
    "wifi_connectivity",
    "staff_service",
    "recommended",
]
for col in rating_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")



1. Đang dọn dẹp cột thừa và chuẩn hóa kiểu dữ liệu...


In [15]:
# ==============================================================================
# BƯỚC 2: TẠO CỘT PHÂN CHIA GIAI ĐOẠN DỮ LIỆU (DATA ERA)
# ==============================================================================
df["data_era"] = df["date_submitted"].apply(
    lambda x: "Modern (2015-Present)"
    if pd.notnull(x) and x.year >= 2015
    else "Historical (Pre-2015)"
)

# ==============================================================================
# BƯỚC 3: CHUẨN HÓA CỘT TEXT (LOUNGE NAME, AIRPORT, TYPE, NATIONALITY)
# ==============================================================================
print(
    "2. Đang chuẩn hóa tên phòng chờ, sân bay, loại phòng chờ và quốc tịch..."
)

# Xóa khoảng trắng thừa và ký tự lạ trong tên phòng chờ và sân bay
for text_col in ["lounge_name", "airport", "type_of_lounge"]:
    if text_col in df.columns:
        df[text_col] = (
            df[text_col].astype(str).str.strip().replace("nan", "Unknown")
        )


# Chuẩn hóa quốc tịch
def clean_nationality(val):
    if pd.isna(val):
        return "Unknown"
    clean_val = str(val).strip().lower()
    return str(val).strip().title() if clean_val in valid_countries else "Unknown"


if "nationality" in df.columns:
    df["nationality"] = df["nationality"].apply(clean_nationality)



2. Đang chuẩn hóa tên phòng chờ, sân bay, loại phòng chờ và quốc tịch...


In [16]:
# ==============================================================================
# BƯỚC 4: PHÂN TÍCH CẢM XÚC (VADER SENTIMENT ANALYSIS)
# ==============================================================================
print("3. Đang chấm điểm cảm xúc cho review phòng chờ...")


def get_sentiment(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Neutral", 0.0
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "Positive", score
    elif score <= -0.05:
        return "Negative", score
    else:
        return "Neutral", score


sentiment_res = df["review"].apply(get_sentiment)
df["sentiment_label"] = [res[0] for res in sentiment_res]
df["sentiment_score"] = [res[1] for res in sentiment_res]



3. Đang chấm điểm cảm xúc cho review phòng chờ...


In [17]:
# ==============================================================================
# BƯỚC 5: TÁCH TỪ KHÓA PHÒNG CHỜ & XÓA REVIEW GỐC
# ==============================================================================
print("4. Đang trích xuất từ khóa F&B / dịch vụ và giảm dung lượng file...")


def extract_lounge_keywords(text):
    if pd.isna(text):
        return ""
    words = re.findall(r"\b[a-z]{3,}\b", str(text).lower())
    keywords = [w for w in words if w not in stop_words]
    return " ".join(keywords)


df["clean_keywords"] = df["review"].apply(extract_lounge_keywords)

# Xóa cột văn bản thô để giảm tối đa dung lượng tải vào VertiPaq Engine của PBI
df = df.drop(columns=["review"])



4. Đang trích xuất từ khóa F&B / dịch vụ và giảm dung lượng file...


In [18]:
import re
import pandas as pd

# Giả lập đọc dữ liệu từ file csv (Trong script chính bạn đã có dòng này)
# df = pd.read_csv("lounge_reviews.csv")


def audit_and_clean_lounge_name(val):
    # 1. Kiểm tra Null / NaN
    if pd.isna(val) or str(val).strip() == "":
        return "Unknown Lounge", "Null/Empty"

    # Chuẩn hóa ban đầu: Xóa khoảng trắng thừa ở 2 đầu và giữa các từ
    clean_val = " ".join(str(val).split()).strip()

    # 2. Kiểm tra độ dài bất thường (Quá ngắn hoặc Quá dài)
    if len(clean_val) < 3:
        return (
            f"Unknown ({clean_val})",
            "Too Short (Invalid Format)",
        )
    if len(clean_val) > 80:
        # Nếu quá dài, có thể văn bản review bị tràn vào cột này -> Cắt lấy 50 ký tự đầu
        return clean_val[:50] + "...", "Too Long (Possible Text Overflow)"

    # 3. Kiểm tra ký tự đặc biệt bất thường hoặc lỗi encoding (Chỉ cho phép Chữ, Số, Khoảng trắng, &, -, /, ')
    # Ví dụ tên hợp lệ: "SilverKris Lounge - T3", "JFK Terminal 7", "Cathay Pacific The Wing / The Pier"
    if re.search(r"[^a-zA-Z0-9\s\&\-\/\.\']", clean_val):
        # Loại bỏ các ký tự rác không mong muốn
        clean_val = re.sub(r"[^a-zA-Z0-9\s\&\-\/\.\']", "", clean_val)
        return clean_val.strip(), "Special Characters Cleaned"

    # 4. Kiểm tra dữ liệu rác bị nhầm sang cột này (Ví dụ chỉ gõ mỗi số hoặc chữ "Lounge" chung chung)
    if clean_val.lower() in [
        "lounge",
        "airport lounge",
        "business lounge",
        "first class lounge",
        "unknown",
        "na",
        "n/a",
    ]:
        return str(val).title(), "Generic Name (Lack of Specificity)"

    # Nếu vượt qua tất cả các bài kiểm tra -> Format chuẩn xác
    return str(val).strip(), "Valid Format"


print("Đang kiểm tra định dạng của cột lounge_name...")

# Áp dụng hàm audit để lấy kết quả làm sạch và nhãn báo cáo lỗi
results = df["lounge_name"].apply(audit_and_clean_lounge_name)

# Gán lại giá trị đã làm sạch vào DataFrame
df["lounge_name"] = [res[0] for res in results]
df["lounge_name_status"] = [res[1] for res in results]

# -------------------------------------------------------------------------
# IN BÁO CÁO KIỂM TRA CHẤT LƯỢNG (DATA AUDIT REPORT)
# -------------------------------------------------------------------------
print("-" * 60)
print("BÁO CÁO TÌNH TRẠNG FORMAT CỦA CỘT LOUNGE_NAME:")
print(df["lounge_name_status"].value_counts())
print("-" * 60)

# Hiển thị các mẫu bị lỗi hoặc cần lưu ý (Nếu có)
invalid_samples = df[df["lounge_name_status"] != "Valid Format"][
    ["lounge_name", "lounge_name_status"]
].drop_duplicates()
if not invalid_samples.empty:
    print("Các mẫu lounge_name đã được chỉnh sửa / phát hiện lỗi format:")
    print(invalid_samples.head(10))
else:
    print("Tuyệt vời! 100% dữ liệu lounge_name đều đúng định dạng chuẩn.")

# (Tùy chọn cho PBI): Xóa cột trạng thái kiểm tra trước khi xuất file CSV
df = df.drop(columns=["lounge_name_status"])

Đang kiểm tra định dạng của cột lounge_name...
------------------------------------------------------------
BÁO CÁO TÌNH TRẠNG FORMAT CỦA CỘT LOUNGE_NAME:
lounge_name_status
Valid Format                          2582
Generic Name (Lack of Specificity)    2456
Special Characters Cleaned              38
Too Short (Invalid Format)              11
Name: count, dtype: int64
------------------------------------------------------------
Các mẫu lounge_name đã được chỉnh sửa / phát hiện lỗi format:
                                           lounge_name  \
2                                              Unknown   
6                                         Unknown (T2)   
37                                    Weltbrger Lounge   
39                                           Weltbrger   
107                                 First Class Lounge   
210  AIR NEW ZEALAND BUSINESS CLASS INT'L LOUNGE RE...   
255                                      Amde Maingard   
260                                 Salon

In [19]:
# Mẹo: Tách lấy tên thương hiệu chính (Cắt bỏ các chỉ dẫn Terminal, Gate phía sau)
# Ví dụ: "SilverKris Lounge T3" -> "SilverKris Lounge"
df["lounge_brand"] = (
    df["lounge_name"]
    .str.split(r"(\s-\s|\s/\s|\sTerminal|\sT\d|\sGate)", regex=True)
    .str[0]
    .str.strip()
)

In [20]:
# ==============================================================================
# BƯỚC 6: XUẤT FILE HOÀN THIỆN CHO POWER BI
# ==============================================================================
output_filename = "lounge_reviews_PBI_Master_Optimized.csv"
df.to_csv(output_filename, index=False)
print("-" * 60)
print(f"XỬ LÝ HOÀN TẤT! File sẵn sàng cho Power BI: '{output_filename}'")
print(f"Kích thước cuối cùng: {df.shape}")
print("-" * 60)
print("Thống kê nhanh các Loại phòng chờ (Type of Lounge) phổ biến nhất:")
if "type_of_lounge" in df.columns:
    print(df["type_of_lounge"].value_counts().head(5))

------------------------------------------------------------
XỬ LÝ HOÀN TẤT! File sẵn sàng cho Power BI: 'lounge_reviews_PBI_Master_Optimized.csv'
Kích thước cuối cùng: (5087, 22)
------------------------------------------------------------
Thống kê nhanh các Loại phòng chờ (Type of Lounge) phổ biến nhất:
type_of_lounge
Business Class    3634
First Class        501
Frequent Flyer     412
Unknown            304
Members            135
Name: count, dtype: int64


In [21]:
def final_quality_check(df, file_name):
    print(f"=== KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG: {file_name} ===")

    # 1. Kiểm tra xem còn cột nào bị NULL/NaN không
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if not null_cols.empty:
        print("⚠️ Cảnh báo: Vẫn còn cột chứa giá trị NULL:")
        for col, count in null_cols.items():
            print(f"   - {col}: trống {count} dòng ({count/len(df)*100:.1f}%)")
    else:
        print("✔️ Tuyệt vời! Không còn ô NULL nào trong dữ liệu.")

    # 2. Kiểm tra kiểu dữ liệu của date_submitted và recommended
    if "date_submitted" in df.columns:
        print(f"✔️ Kiểu dữ liệu ngày tháng: {df['date_submitted'].dtype}")
    if "recommended" in df.columns:
        print(
            f"✔️ Kiểu dữ liệu NPS (recommended): {df['recommended'].dtype} (Giá trị mẫu: {df['recommended'].unique()})"
        )
    print("-" * 50)


# Ví dụ cách dùng:
final_quality_check(df, "Lounge Reviews")

=== KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG: Lounge Reviews ===
⚠️ Cảnh báo: Vẫn còn cột chứa giá trị NULL:
   - type_of_traveller: trống 1453 dòng (28.6%)
   - comfort: trống 24 dòng (0.5%)
   - cleanliness: trống 28 dòng (0.6%)
   - bar_and_beverages: trống 152 dòng (3.0%)
   - catering: trống 88 dòng (1.7%)
   - washrooms: trống 626 dòng (12.3%)
   - wifi_connectivity: trống 443 dòng (8.7%)
   - staff_service: trống 117 dòng (2.3%)
✔️ Kiểu dữ liệu ngày tháng: datetime64[ns]
✔️ Kiểu dữ liệu NPS (recommended): int64 (Giá trị mẫu: [1 0])
--------------------------------------------------


In [22]:
# ==============================================================================
# BƯỚC 4: XÓA CỘT THỪA & CHUẨN HÓA KIỂU DỮ LIỆU
# ==============================================================================
print("1. Đang dọn dẹp cột thừa và chuẩn hóa kiểu dữ liệu...")
cols_to_drop = ["customer_name", "updated_at", "date_visit"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Chuẩn hóa ngày tháng gửi review
df["date_submitted"] = pd.to_datetime(df["date_submitted"], errors="coerce")

# Chuẩn hóa TOÀN BỘ 7 cột điểm số phòng chờ + recommended về kiểu Numeric
rating_cols = [
    "comfort",
    "cleanliness",
    "bar_and_beverages",
    "catering",
    "washrooms",
    "wifi_connectivity",
    "staff_service",
    "recommended",
]
for col in rating_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ==============================================================================
# BƯỚC 5: LẤP ĐẦY "UNKNOWN" CHO CÁC CỘT PHÂN LOẠI (TEXT) -> TRÁNH LỖI (BLANK) PBI
# ==============================================================================
print("2. Đang chuẩn hóa các cột văn bản và phân loại...")

# Cột phân loại khách hàng và loại phòng chờ -> Điền "Unknown"
text_cols_to_fill = ["type_of_lounge", "type_of_traveller"]
for col in text_cols_to_fill:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace(["nan", "None", "NULL", "", "N/A", "n/a"], "Unknown")
            .fillna("Unknown")
        )

# Xóa khoảng trắng thừa trong cột airport (sân bay)
if "airport" in df.columns:
    df["airport"] = (
        df["airport"].astype(str).str.strip().replace("nan", "Unknown")
    )

# ==============================================================================
# BƯỚC 6: CHUẨN HÓA QUỐC TỊCH & KIỂM TRA FORMAT LOUNGE_NAME
# ==============================================================================
# 6.1. Làm sạch Quốc tịch
def clean_nationality(val):
    if pd.isna(val):
        return "Unknown"
    clean_val = str(val).strip().lower()
    return str(val).strip().title() if clean_val in valid_countries else "Unknown"

if "nationality" in df.columns:
    df["nationality"] = df["nationality"].apply(clean_nationality)

# 6.2. Kiểm tra và làm sạch format Lounge Name
def audit_and_clean_lounge_name(val):
    if pd.isna(val) or str(val).strip() == "":
        return "Unknown Lounge"
    clean_val = " ".join(str(val).split()).strip()
    if len(clean_val) < 3 or clean_val.lower() in ["lounge", "airport lounge", "unknown", "na"]:
        return str(val).title() if len(clean_val) >= 3 else f"Unknown ({clean_val})"
    # Loại bỏ ký tự lạ, giữ lại chữ, số và các dấu cơ bản (& - / ' .)
    clean_val = re.sub(r"[^a-zA-Z0-9\s\&\-\/\.\']", "", clean_val)
    return clean_val.strip()

if "lounge_name" in df.columns:
    df["lounge_name"] = df["lounge_name"].apply(audit_and_clean_lounge_name)

# ==============================================================================
# BƯỚC 7: XÓA DÒNG RÁC (KHÁCH KHÔNG CHẤM BẤT KỲ ĐIỂM SỐ NÀO)
# ==============================================================================
print("3. Đang kiểm tra và lọc bỏ các bài review rác...")
lounge_ratings = [
    "comfort", "cleanliness", "bar_and_beverages", 
    "catering", "washrooms", "wifi_connectivity", "staff_service"
]
existing_ratings = [col for col in lounge_ratings if col in df.columns]

if existing_ratings:
    initial_rows = len(df)
    df = df.dropna(subset=existing_ratings, how='all')
    deleted_rows = initial_rows - len(df)
    if deleted_rows > 0:
        print(f"✔️ Đã xóa {deleted_rows} dòng review rác (khách không chấm điểm dịch vụ nào).")

# ==============================================================================
# BƯỚC 8: PHÂN TÍCH CẢM XÚC (VADER) & TÁCH KEYWORDS CHO WORD CLOUD
# ==============================================================================
print("4. Đang chấm điểm cảm xúc (Sentiment) & trích xuất từ khóa sạch...")

def get_sentiment(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Neutral", 0.0
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "Positive", score
    elif score <= -0.05:
        return "Negative", score
    else:
        return "Neutral", score

# Chấm điểm trên text gốc
sentiment_res = df["review"].apply(get_sentiment)
df["sentiment_label"] = [res[0] for res in sentiment_res]
df["sentiment_score"] = [res[1] for res in sentiment_res]

# Trích xuất từ khóa sạch (chỉ lấy từ >= 3 chữ cái, không nằm trong stop_words)
def extract_lounge_keywords(text):
    if pd.isna(text):
        return ""
    words = re.findall(r"\b[a-z]{3,}\b", str(text).lower())
    keywords = [w for w in words if w not in stop_words]
    return " ".join(keywords)

df["clean_keywords"] = df["review"].apply(extract_lounge_keywords)

# [QUAN TRỌNG NHẤT CHO PBI]: Xóa cột review gốc để giảm 80% dung lượng
df = df.drop(columns=["review"])

# ==============================================================================
# BƯỚC 9: KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG & XUẤT FILE PBI
# ==============================================================================
print("\n--- BÁO CÁO KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG ---")
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]
if not null_cols.empty:
    print("⚠️ Chú ý: Các cột sau vẫn còn chứa giá trị NULL (Khuyến nghị: CHỈ NÊN LÀ CÁC CỘT ĐIỂM SỐ):")
    for col, count in null_cols.items():
        print(f"   - {col}: trống {count} dòng ({count/len(df)*100:.1f}%)")
else:
    print("✔️ Tuyệt vời! Không còn ô NULL nào trong toàn bộ dữ liệu.")

# Lưu file đầu ra
output_filename = DATA_RAW / "lounge_reviews_PBI_Master_Optimized.csv"
df.to_csv(output_filename, index=False)
print("=" * 60)
print(f"🎉 XỬ LÝ HOÀN TẤT! File đã lưu tại: {output_filename}")
print(f"📐 Kích thước cuối cùng sẵn sàng lên Power BI: {df.shape}")
print("=" * 60)

1. Đang dọn dẹp cột thừa và chuẩn hóa kiểu dữ liệu...
2. Đang chuẩn hóa các cột văn bản và phân loại...
3. Đang kiểm tra và lọc bỏ các bài review rác...
✔️ Đã xóa 16 dòng review rác (khách không chấm điểm dịch vụ nào).
4. Đang chấm điểm cảm xúc (Sentiment) & trích xuất từ khóa sạch...


KeyError: 'review'

---

## Seat Reviews — Cleaning

*Source notebook: `notebook_seat_reviews.ipynb`*


In [24]:
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pandas as pd
import pycountry

# ==============================================================================
# BƯỚC 0: TẢI DỮ LIỆU & CHUẨN BỊ THƯ VIỆN NLP
# ==============================================================================
nltk.download("vader_lexicon", quiet=True)
sia = SentimentIntensityAnalyzer()

# Đọc file dữ liệu đánh giá ghế ngồi
df = pd.read_csv(DATA_RAW / "seat_reviews.csv")
print(f"Kích thước file gốc: {df.shape}")

# 1. Từ điển Stopwords tiếng Anh nền tảng
base_stopwords = {
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "your",
    "yours",
    "he",
    "him",
    "his",
    "she",
    "her",
    "it",
    "its",
    "they",
    "them",
    "their",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "can",
    "will",
    "just",
    "don",
    "should",
    "now",
}

# 2. Từ điển Stopwords ĐẶC THÙ CHO GHẾ NGỒI (Seat-specific Stopwords)
seat_noise_words = {
    "seat",
    "seats",
    "seated",
    "seating",
    "row",
    "rows",
    "sit",
    "sat",
    "sitting",
    "class",
    "economy",
    "business",
    "first",
    "premium",
    "cabin",
    "flight",
    "flights",
    "fly",
    "flying",
    "flew",
    "plane",
    "planes",
    "aircraft",
    "airline",
    "airlines",
    "time",
    "times",
    "hour",
    "hours",
    "minute",
    "minutes",
    "passenger",
    "passengers",
    "people",
    "traveller",
    "would",
    "could",
    "get",
    "got",
    "getting",
    "one",
    "two",
    "three",
    "also",
    "even",
    "really",
    "way",
    "make",
    "made",
    "much",
    "bit",
    "quite",
    "well",
    "take",
    "took",
    "taking",
    "give",
    "gave",
    "go",
    "going",
    "went",
    "say",
    "said",
    "told",
    "see",
    "saw",
    "seen",
    "know",
    "think",
    "thought",
    "want",
    "wanted",
    "need",
    "needed",
    "try",
    "tried",
    "still",
    "like",
    "good",
    "bad",  # Loại bỏ good/bad chung chung để lòi ra: cramped, hard, spacious, uncomfortable, broken
}
stop_words = base_stopwords.union(seat_noise_words)

# 3. Chuẩn bị tập từ điển quốc gia chuẩn ISO
valid_countries = {c.name.lower() for c in pycountry.countries}
for c in pycountry.countries:
    if hasattr(c, "official_name"):
        valid_countries.add(c.official_name.lower())
    if hasattr(c, "common_name"):
        valid_countries.add(c.common_name.lower())
custom_countries = {
    "uk",
    "usa",
    "united states",
    "united kingdom",
    "uae",
    "russia",
    "south korea",
    "north korea",
    "vietnam",
    "laos",
    "taiwan",
    "hong kong",
    "macau",
}
valid_countries = valid_countries.union(custom_countries)



Kích thước file gốc: (3766, 26)


In [25]:
def clean_seat_layout(val):
    if pd.isna(val):
        return "Unknown"

    # Chuyển về chữ thường và xóa sạch các khoảng trắng thừa
    val_str = str(val).lower().replace(" ", "").strip()

    # -------------------------------------------------------------------------
    # BƯỚC 1: XỬ LÝ LỖI NGÀY THÁNG DO EXCEL / SCRAPING TỰ ĐỘNG ĐỔI
    # Ví dụ: "2-4-2002" -> "2-4-02", "3-3-2003" -> "3-3-03"
    # -------------------------------------------------------------------------
    # Nhận diện các cụm số kết thúc bằng năm 20xx (2001 đến 2009)
    date_err_pattern = r"^(\d+)[\-\/x](\d+)[\-\/x]20(\d{2})$"
    if re.match(date_err_pattern, val_str):
        # Biến "2-4-2002" lại thành "2x4x2"
        val_str = re.sub(date_err_pattern, r"\1x\2x\3", val_str)

    # Nhận diện trường hợp số 0 ở đầu do format ngày (ví dụ: "02-04-02" -> "2x4x2")
    val_str = re.sub(r"^0+(\d)", r"\1", val_str)
    val_str = re.sub(r"x0+(\d)", r"x\1", val_str)
    val_str = re.sub(r"\-0+(\d)", r"-\1", val_str)

    # -------------------------------------------------------------------------
    # BƯỚC 2: CHUẨN HÓA KÝ TỰ PHÂN CÁCH VỀ CHỮ 'x'
    # Biến tất cả dấu gạch ngang (-), gạch chéo (/), dấu chấm (.) thành 'x'
    # -------------------------------------------------------------------------
    val_str = re.sub(r"[\-\/\._]", "x", val_str)

    # -------------------------------------------------------------------------
    # BƯỚC 3: KIỂM TRA TÍNH HỢP LỆ CỦA SƠ ĐỒ GHẾ
    # Sơ đồ chuẩn thường là dạng 2 số (3x3, 2x2) hoặc 3 số (2x4x2, 3x3x3, 1x2x1)
    # -------------------------------------------------------------------------
    valid_layout_pattern = r"^\d+x\d+(x\d+)?$"
    if re.match(valid_layout_pattern, val_str):
        return val_str
    else:
        # Nếu khách ghi chữ lung tung không phải sơ đồ ghế -> Trả về Unknown
        return "Unknown"


# Áp dụng hàm làm sạch mới vào DataFrame
print("Đang xử lý và chuẩn hóa sơ đồ ghế (seat_layout)...")
if "seat_layout" in df.columns:
    df["seat_layout"] = df["seat_layout"].apply(clean_seat_layout)

# Kiểm tra kết quả các sơ đồ ghế phổ biến nhất sau làm sạch
print("-" * 50)
print("Thống kê top 10 Sơ đồ ghế chuẩn xác sau khi xử lý:")
print(df["seat_layout"].value_counts().head(10))
print("-" * 50)

Đang xử lý và chuẩn hóa sơ đồ ghế (seat_layout)...
--------------------------------------------------
Thống kê top 10 Sơ đồ ghế chuẩn xác sau khi xử lý:
seat_layout
3x3        919
2x4x2      852
3x4x3      762
3x3x3      575
2x3x2      222
2x2        130
1x2x1       97
2x2x2       76
Unknown     65
2x5x2       20
Name: count, dtype: int64
--------------------------------------------------


In [26]:
#==============================================================================
# BƯỚC 1: XÓA CỘT THỪA & ĐỔI DATA TYPE
# ==============================================================================
print("1. Đang dọn dẹp cột thừa và chuẩn hóa kiểu dữ liệu...")
cols_to_drop = ["customer_name", "updated_at", "date_flown"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Chuẩn hóa ngày tháng
df["date_submitted"] = pd.to_datetime(df["date_submitted"], errors="coerce")

# Chuẩn hóa TOÀN BỘ 12 cột điểm số ghế ngồi + recommended về kiểu Numeric
rating_cols = [
    "seat_legroom",
    "seat_recline",
    "seat_width",
    "aisle_space",
    "seat_storage",
    "power_supply",
    "viewing_tv_screen",
    "sleep_comfort",
    "sitting_comfort",
    "seat_bed_width",
    "seat_bed_length",
    "seat_privacy",
    "recommended",
]
for col in rating_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ==============================================================================
# BƯỚC 2: TẠO CỘT PHÂN CHIA GIAI ĐOẠN DỮ LIỆU (DATA ERA)
# ==============================================================================
df["data_era"] = df["date_submitted"].apply(
    lambda x: "Modern (2015-Present)"
    if pd.notnull(x) and x.year >= 2015
    else "Historical (Pre-2015)"
)


# ==============================================================================
# BƯỚC 3: CHUẨN HÓA MÁY BAY, SƠ ĐỒ GHẾ & QUỐC TỊCH
# ==============================================================================
# 3.1. Gom nhóm máy bay
def clean_aircraft(val):
    if pd.isna(val):
        return "Unknown"
    val_str = str(val).upper()
    if "/" in val_str or " AND " in val_str or "," in val_str:
        return "Mixed Fleet"
    if "BOEING" in val_str or re.search(r"\bB?7[0-9]{2}\b", val_str):
        match = re.search(r"7[0-9]{2}", val_str)
        return f"Boeing {match.group(0)}" if match else "Boeing (Other)"
    elif "AIRBUS" in val_str or re.search(r"\bA3[0-9]{2}\b", val_str):
        match = re.search(r"3[0-9]{2}", val_str)
        return f"Airbus A{match.group(0)}" if match else "Airbus (Other)"
    else:
        return "Other Aircraft"


print("2. Đang chuẩn hóa loại máy bay, sơ đồ ghế và quốc tịch...")
if "aircraft_type" in df.columns:
    df["aircraft_type"] = df["aircraft_type"].apply(clean_aircraft)

# 3.2. Làm sạch sơ đồ ghế (seat_layout)
if "seat_layout" in df.columns:
    df["seat_layout"] = (
        df["seat_layout"]
        .astype(str)
        .str.lower()
        .str.replace(" ", "")
        .replace("nan", "Unknown")
    )


# 3.3. Làm sạch quốc tịch
def clean_nationality(val):
    if pd.isna(val):
        return "Unknown"
    clean_val = str(val).strip().lower()
    return str(val).strip().title() if clean_val in valid_countries else "Unknown"


if "nationality" in df.columns:
    df["nationality"] = df["nationality"].apply(clean_nationality)




1. Đang dọn dẹp cột thừa và chuẩn hóa kiểu dữ liệu...
2. Đang chuẩn hóa loại máy bay, sơ đồ ghế và quốc tịch...


In [27]:
# ==============================================================================
# BƯỚC 4: PHÂN TÍCH CẢM XÚC (VADER SENTIMENT ANALYSIS)
# ==============================================================================
print("3. Đang chấm điểm cảm xúc cho trải nghiệm ghế ngồi...")


def get_sentiment(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Neutral", 0.0
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "Positive", score
    elif score <= -0.05:
        return "Negative", score
    else:
        return "Neutral", score


sentiment_res = df["review"].apply(get_sentiment)
df["sentiment_label"] = [res[0] for res in sentiment_res]
df["sentiment_score"] = [res[1] for res in sentiment_res]


3. Đang chấm điểm cảm xúc cho trải nghiệm ghế ngồi...


In [28]:
# ==============================================================================
# BƯỚC 5: TÁCH TỪ KHÓA GHẾ NGỒI & XÓA REVIEW GỐC (TỐI ƯU VERTIPAQ PBI)
# ==============================================================================
print("4. Đang trích xuất từ khóa sạch và giảm dung lượng file...")


def extract_seat_keywords(text):
    if pd.isna(text):
        return ""
    words = re.findall(r"\b[a-z]{3,}\b", str(text).lower())
    keywords = [w for w in words if w not in stop_words]
    return " ".join(keywords)


df["clean_keywords"] = df["review"].apply(extract_seat_keywords)

# Xóa cột văn bản thô để giảm dung lượng tải vào Power BI
df = df.drop(columns=["review"])

4. Đang trích xuất từ khóa sạch và giảm dung lượng file...


In [29]:
# Danh sách các cột phân loại (Categorical Columns) cần lấp đầy bằng 'Unknown'
categorical_cols = [
    "type_of_traveller",
    "seat_type",
    "type_of_lounge",
    "experience_at_airport",
    "aircraft_type",
    "seat_layout",
]

# Vòng lặp tự động tìm và làm sạch các cột có mặt trong file hiện tại
for col in categorical_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace(["nan", "None", "NULL", "", "N/A", "n/a"], "Unknown")
            .fillna("Unknown")
        )
        print(f"✔️ Đã chuẩn hóa 'Unknown' cho cột: {col}")

✔️ Đã chuẩn hóa 'Unknown' cho cột: type_of_traveller
✔️ Đã chuẩn hóa 'Unknown' cho cột: seat_type
✔️ Đã chuẩn hóa 'Unknown' cho cột: aircraft_type
✔️ Đã chuẩn hóa 'Unknown' cho cột: seat_layout


In [30]:
# ==============================================================================
# BƯỚC 6: XUẤT FILE HOÀN THIỆN CHO POWER BI
# ==============================================================================
output_filename = "seat_reviews_PBI_Master_Optimized.csv"
df.to_csv(output_filename, index=False)
print("-" * 60)
print(f"XỬ LÝ HOÀN TẤT! File sẵn sàng cho Power BI: '{output_filename}'")
print(f"Kích thước cuối cùng: {df.shape}")
print("-" * 60)
print("Thống kê nhanh Sơ đồ ghế (Seat Layout) phổ biến nhất:")
if "seat_layout" in df.columns:
    print(df["seat_layout"].value_counts().head(5))

------------------------------------------------------------
XỬ LÝ HOÀN TẤT! File sẵn sàng cho Power BI: 'seat_reviews_PBI_Master_Optimized.csv'
Kích thước cuối cùng: (3766, 26)
------------------------------------------------------------
Thống kê nhanh Sơ đồ ghế (Seat Layout) phổ biến nhất:
seat_layout
3x3      919
2x4x2    852
3x4x3    762
3x3x3    575
2x3x2    222
Name: count, dtype: int64


In [31]:
def final_quality_check(df, file_name):
    print(f"=== KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG: {file_name} ===")

    # 1. Kiểm tra xem còn cột nào bị NULL/NaN không
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if not null_cols.empty:
        print("⚠️ Cảnh báo: Vẫn còn cột chứa giá trị NULL:")
        for col, count in null_cols.items():
            print(f"   - {col}: trống {count} dòng ({count/len(df)*100:.1f}%)")
    else:
        print("✔️ Tuyệt vời! Không còn ô NULL nào trong dữ liệu.")

    # 2. Kiểm tra kiểu dữ liệu của date_submitted và recommended
    if "date_submitted" in df.columns:
        print(f"✔️ Kiểu dữ liệu ngày tháng: {df['date_submitted'].dtype}")
    if "recommended" in df.columns:
        print(
            f"✔️ Kiểu dữ liệu NPS (recommended): {df['recommended'].dtype} (Giá trị mẫu: {df['recommended'].unique()})"
        )
    print("-" * 50)


# Ví dụ cách dùng:
final_quality_check(df, "Seat Reviews")

=== KIỂM TRA CHẤT LƯỢNG CUỐI CÙNG: Seat Reviews ===
⚠️ Cảnh báo: Vẫn còn cột chứa giá trị NULL:
   - seat_legroom: trống 223 dòng (5.9%)
   - seat_recline: trống 224 dòng (5.9%)
   - seat_width: trống 223 dòng (5.9%)
   - aisle_space: trống 224 dòng (5.9%)
   - seat_storage: trống 1318 dòng (35.0%)
   - power_supply: trống 2382 dòng (63.3%)
   - viewing_tv_screen: trống 980 dòng (26.0%)
   - sleep_comfort: trống 3545 dòng (94.1%)
   - sitting_comfort: trống 3543 dòng (94.1%)
   - seat_bed_width: trống 3544 dòng (94.1%)
   - seat_bed_length: trống 3546 dòng (94.2%)
   - seat_privacy: trống 3545 dòng (94.1%)
✔️ Kiểu dữ liệu ngày tháng: datetime64[ns]
✔️ Kiểu dữ liệu NPS (recommended): int64 (Giá trị mẫu: [0 1])
--------------------------------------------------
